# Denoising Diffusion Probabilistic Model (DDPM)

denoise 的模型共享，区别在于会有去噪步骤的额外输入。
<img src="../../assets/denoise_steps.png">

denoise 内部使用一个 noise predictor 预测输入图像的噪声，然后将噪声与输入图像相减，得到去噪后的图像。
<img src="../../assets/denoise.png">

noise predictor 的训练：

通过加噪过程 (diffusion progress) 得到不同步骤的噪声-图像对，将此作为训练数据
<img src="../../assets/diffusion_progress.png">

ddpm 算法：
<img src="../../assets/ddpm_algorithm.png">

In [1]:
import torch
from torch.optim import Adam
import torch.nn.functional as F
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader
from torchvision import transforms
from datasets import load_dataset
from transformers import CLIPTokenizer, CLIPTextModel
from PIL import Image
import numpy as np
import os
from tqdm.auto import tqdm
from ddpm import UNet, NoiseScheduler, sample

C:\Users\27344\.conda\envs\AI\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# hyperparameters
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
image_size = 64
in_channels = 3
epochs = 100
batch_size = 32
lr = 1e-4
T = 1000
save_checkpoint = 10

In [3]:
def transform_all(data):
    transform = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ])

    images = [transform(image.convert('RGB')) for image in data["image"]]
    en_texts = data["en_text"]
    return {"image": images, "en_text": en_texts}

In [4]:
# load dataset
dataset = load_dataset("svjack/pokemon-blip-captions-en-zh", split="train", cache_dir=r"D:\HuggingFace\cache")
dataset.set_transform(transform_all)
train_dataset = dataset.select(range(600))
val_dataset = dataset.select(range(600, 800))
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0, drop_last=True)

In [5]:
# load clip model
tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-base-patch32")
text_encoder = CLIPTextModel.from_pretrained("openai/clip-vit-base-patch32").to(device)

In [6]:
# log
save_dir = "../logs/ddpm"
os.makedirs(save_dir, exist_ok=True)

In [7]:
# define model
diffusion_model = UNet(in_channels=in_channels).to(device)
noise_scheduler = NoiseScheduler(T, device)

optimizer = Adam(diffusion_model.parameters(), lr=lr, weight_decay=5e-4)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10, min_lr=5e-5)

In [8]:
# train
for epoch in range(epochs):
    print(f"Epoch {epoch + 1}/{epochs}")
    diffusion_model.train()
    train_bar = tqdm(total=len(train_loader), desc="Training")
    train_loss = 0.
    for batch in train_loader:
        images = batch['image'].to(device)
        text = batch['en_text']
        text_input = tokenizer(text, padding="max_length", max_length=tokenizer.model_max_length, truncation=True, return_tensors="pt")
        text_embeddings = text_encoder(text_input["input_ids"].to(device)).last_hidden_state

        t = torch.randint(0, T, (images.shape[0],), device=device).long()
        noisy_images, noise = noise_scheduler.add_noise(images, t)
        noise_pred = diffusion_model(noisy_images, t, text_embeddings)
        loss = F.mse_loss(noise_pred, noise)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        train_bar.update(1)
        train_bar.set_postfix({"loss": loss.item()})

    diffusion_model.eval()
    val_bar = tqdm(total=len(val_loader), desc="Validation")
    val_loss = 0.
    with torch.no_grad():
        for batch in val_loader:
            images = batch['image'].to(device)
            text = batch['en_text']
            text_input = tokenizer(text, padding="max_length", max_length=tokenizer.model_max_length, truncation=True, return_tensors="pt")
            text_embeddings = text_encoder(text_input["input_ids"].to(device)).last_hidden_state

            t = torch.randint(0, T, (images.shape[0],), device=device).long()
            noisy_images, noise = noise_scheduler.add_noise(images, t)
            noise_pred = diffusion_model(noisy_images, t, text_embeddings)
            loss = F.mse_loss(noise_pred, noise)

            val_loss += loss.item()
            val_bar.update(1)
            val_bar.set_postfix({"loss": loss.item()})
    scheduler.step(val_loss)

    print(f"Epoch {epoch + 1}/{epochs}:\n\tTrain Loss: {train_loss / len(train_loader)}\n\tVal Loss: {val_loss / len(val_loader)}")

    if (epoch + 1) % save_checkpoint == 0:
        # save model
        torch.save({
            "epoch": epoch + 1,
            "model_state_dict": diffusion_model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "train_loss": train_loss,
            "val_loss": val_loss,
        }, os.path.join(save_dir, f"diffusion_{epoch + 1}.pth"))
        print(f"Model saved at epoch {epoch + 1}...")
        # save images
        diffusion_model.eval()
        with torch.no_grad():
            sample_text = ["a red pokémon with a red fire tail"]
            text_input = tokenizer(sample_text, padding="max_length", max_length=tokenizer.model_max_length, truncation=True, return_tensors="pt")
            text_embeddings = text_encoder(text_input["input_ids"].to(device)).last_hidden_state
            epsilon = torch.randn(len(sample_text), in_channels, image_size, image_size).to(device)
            sampled_images = sample(diffusion_model, epsilon, noise_scheduler,text_embeddings, guidance_scale=3.0)
            for i, image in enumerate(sampled_images):
                image = (image + 1) / 2  # denormalize
                image = image.detach().cpu().permute(1, 2, 0).numpy()
                image = (image * 255).astype(np.uint8)
                image_pil = Image.fromarray(image)
                image_pil.save(os.path.join(save_dir, f"image_epoch{epoch + 1}_sample_{i}.png"))
torch.save({
    "model": diffusion_model.state_dict(),
    "optimizer": optimizer.state_dict(),
    "scheduler": scheduler.state_dict(),
}, os.path.join(save_dir, "diffusion_final.pth"))

Epoch 1/100


Validation: 100%|██████████| 6/6 [00:03<00:00,  1.99it/s, loss=0.219]

Epoch 1/100:
	Train Loss: 0.3495645854208205
	Val Loss: 0.2760440806547801
Epoch 2/100




Training: 100%|██████████| 18/18 [00:17<00:00,  1.06it/s, loss=0.263]


Training:   6%|▌         | 1/18 [00:00<00:13,  1.28it/s]

Training:   6%|▌         | 1/18 [00:00<00:13,  1.28it/s, loss=0.271]

Training:  11%|█         | 2/18 [00:01<00:12,  1.32it/s, loss=0.271]

Training:  11%|█         | 2/18 [00:01<00:12,  1.32it/s, loss=0.372]

Training:  17%|█▋        | 3/18 [00:02<00:11,  1.36it/s, loss=0.372]

Training:  17%|█▋        | 3/18 [00:02<00:11,  1.36it/s, loss=0.279]

Training:  22%|██▏       | 4/18 [00:02<00:10,  1.37it/s, loss=0.279]

Training:  22%|██▏       | 4/18 [00:02<00:10,  1.37it/s, loss=0.314]

Training:  28%|██▊       | 5/18 [00:03<00:09,  1.35it/s, loss=0.314]

Training:  28%|██▊       | 5/18 [00:03<00:09,  1.35it/s, loss=0.149]

Training:  33%|███▎      | 6/18 [00:04<00:08,  1.34it/s, loss=0.149]

Training:  33%|███▎      | 6/18 [00:04<00:08,  1.34it/s, loss=0.274]

Training:  39%|███▉      | 7/18 [00:05<00:08,  1.34it/s, loss=0.274]

Training:  39%|███▉      | 7

Epoch 2/100:
	Train Loss: 0.2541888993647363
	Val Loss: 0.2832181702057521
Epoch 3/100


...


Training: 100%|██████████| 18/18 [00:15<00:00,  1.20it/s, loss=0.19]

Training: 100%|██████████| 18/18 [00:13<00:00,  1.36it/s, loss=0.0927]

Validation: 100%|██████████| 6/6 [00:15<00:00,  2.63s/it, loss=0.178]


Validation:  17%|█▋        | 1/6 [00:00<00:02,  1.92it/s]

Validation:  17%|█▋        | 1/6 [00:00<00:02,  1.92it/s, loss=0.109]

Validation:  33%|███▎      | 2/6 [00:00<00:01,  2.26it/s, loss=0.109]

Validation:  33%|███▎      | 2/6 [00:00<00:01,  2.26it/s, loss=0.0982]

Validation:  50%|█████     | 3/6 [00:01<00:01,  2.03it/s, loss=0.0982]

Validation:  50%|█████     | 3/6 [00:01<00:01,  2.03it/s, loss=0.172] 

Validation:  67%|██████▋   | 4/6 [00:01<00:00,  2.13it/s, loss=0.172]

Validation:  67%|██████▋   | 4/6 [00:01<00:00,  2.13it/s, loss=0.106]

Validation:  83%|████████▎ | 5/6 [00:02<00:00,  1.97it/s, loss=0.106]

Validation:  83%|████████▎ | 5/6 [00:02<00:00,  1.97it/s, loss=0.148]

Validation: 100%|██████████| 6/6 [00:02<00:00,  2.24it/s, loss=0.148]

Validation: 1

Epoch 9/100:
	Train Loss: 0.12968454758326212
	Val Loss: 0.1312361086408297
Epoch 10/100


Validation: 100%|██████████| 6/6 [00:16<00:00,  2.69s/it, loss=0.154]

Validation: 100%|██████████| 6/6 [00:03<00:00,  2.20it/s, loss=0.113]

Epoch 10/100:
	Train Loss: 0.12038078366054429
	Val Loss: 0.12399975210428238
Model saved at epoch 10...


KeyboardInterrupt: 